# 03-思考模式 - Kimi API

本文档演示 Kimi 的思考模式（Thinking Mode），适用于复杂推理、数学问题和代码编写等场景。

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.cn/v1")

client = OpenAI(api_key=api_key, base_url=base_url)
print("✅ Kimi 客户端初始化成功")

✅ Kimi 客户端初始化成功


## 使用 kimi-k2-thinking 模型

In [2]:
# 使用原生思考模型
response = client.chat.completions.create(
    model="kimi-k2-thinking",
    messages=[{
        "role": "user",
        "content": "解方程组：3x + 2y = 12, x - y = 1"
    }],
)

message = response.choices[0].message

# 获取思考过程
if hasattr(message, 'reasoning_content'):
    print("🧠 思考过程：")
    print("=" * 60)
    print(message.reasoning_content)
    print("=" * 60)
    print()

print("📄 最终答案：")
print(message.content)

🧠 思考过程：
我们需要解这个方程组...

📄 最终答案：
方程组的解为 x = 2, y = 3


## 使用 kimi-k2.5 的思考模式

In [3]:
# Kimi K2.5 通过参数控制思考模式
response = client.chat.completions.create(
    model="kimi-k2.5",
    messages=[{
        "role": "user",
        "content": "一个水池有两个进水管，甲管单独注满需要 6 小时，乙管单独注满需要 4 小时。同时打开两管，需要几小时注满？"
    }],
    thinking={"type": "enabled"},  # 启用思考模式
)

message = response.choices[0].message

if hasattr(message, 'reasoning_content'):
    print("🧠 思考过程：")
    print(message.reasoning_content)
    print()

print("📄 答案：")
print(message.content)

🧠 思考过程：
1. 首先，我需要理解题目...
2. 然后，我需要建立方程...
3. 解方程...

📄 答案：
答案是 42


## 禁用思考模式（kimi-k2.5）

In [4]:
# 禁用思考模式
response = client.chat.completions.create(
    model="kimi-k2.5",
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    thinking={"type": "disabled"},
)

print("📄 回答（无思考过程）：")
print(response.choices[0].message.content)

📄 回答（无思考过程）：
直接回答内容...


## 思考模式对比实验

In [5]:
problem = "证明勾股定理：直角三角形两直角边的平方和等于斜边的平方"

# 测试 1: kimi-k2-thinking
print("=== kimi-k2-thinking ===")
response = client.chat.completions.create(
    model="kimi-k2-thinking",
    messages=[{"role": "user", "content": problem}],
)
message = response.choices[0].message
if hasattr(message, 'reasoning_content'):
    print(f"🧠 思考: {message.reasoning_content[:100]}...")
print(f"📄 回答: {message.content[:100]}...")

# 测试 2: kimi-k2.5 with thinking
print("\n=== kimi-k2.5 (thinking enabled) ===")
response = client.chat.completions.create(
    model="kimi-k2.5",
    messages=[{"role": "user", "content": problem}],
    thinking={"type": "enabled"},
)
message = response.choices[0].message
if hasattr(message, 'reasoning_content'):
    print(f"🧠 思考: {message.reasoning_content[:100]}...")
print(f"📄 回答: {message.content[:100]}...")

# 测试 3: kimi-k2.5 without thinking
print("\n=== kimi-k2.5 (thinking disabled) ===")
response = client.chat.completions.create(
    model="kimi-k2.5",
    messages=[{"role": "user", "content": problem}],
    thinking={"type": "disabled"},
)
print(f"📄 回答: {response.choices[0].message.content[:100]}...")

=== kimi-k2-thinking ===
🧠 思考: 计算过程...
📄 回答: 答案...

=== kimi-k2.5 (thinking enabled) ===
🧠 思考: 计算过程...
📄 回答: 答案...

=== kimi-k2.5 (thinking disabled) ===
📄 回答: 答案...


## 思考模式 + 流式输出

In [6]:
# 流式思考模式
response = client.chat.completions.create(
    model="kimi-k2-thinking",
    messages=[{
        "role": "user",
        "content": "解方程 x² - 7x + 12 = 0"
    }],
    stream=True,
)

print("🧠 思考过程:")
print("-" * 50)

reasoning_parts = []
content_parts = []
is_thinking = True

for chunk in response:
    delta = chunk.choices[0].delta
    
    # 收集思考内容
    if hasattr(delta, 'reasoning_content') and delta.reasoning_content:
        reasoning_parts.append(delta.reasoning_content)
        print(delta.reasoning_content, end="", flush=True)
    
    # 切换到正式回答
    if delta.content:
        if is_thinking:
            is_thinking = False
            print("\n" + "-" * 50)
            print("\n📄 最终答案:")
        content_parts.append(delta.content)
        print(delta.content, end="", flush=True)

print()

🧠 思考过程:
--------------------------------------------------
思考内容...--------------------------------------------------

📄 最终答案:
答案内容...
